In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/shah9212/notebook-ssg/__results__.html
/kaggle/input/notebooks/shah9212/notebook-ssg/__notebook__.ipynb
/kaggle/input/notebooks/shah9212/notebook-ssg/__output__.json
/kaggle/input/notebooks/shah9212/notebook-ssg/results.zip
/kaggle/input/notebooks/shah9212/notebook-ssg/custom.css
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/LICENSE
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/yolov8m.pt
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/.gitignore
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/README.md
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/requirements.txt
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/yolo26n.pt
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/setup.py
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/runs/detect/det/yolov8m_spatial/BoxR_curve.png
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/runs/detect/det/yolov8m_spatial

In [2]:
%%bash
cd /kaggle/working && rm -rf SGG-Benchmark
git clone -q https://github.com/Maelic/SGG-Benchmark.git
cd SGG-Benchmark && pip install -e . -q
pip install -q ultralytics hydra-core omegaconf
echo "INSTALL DONE"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 10.1 MB/s eta 0:00:00
INSTALL DONE


In [3]:
%%bash
set -e
BASE=/kaggle/working/SGG-Benchmark
INPUT=$(dirname $(find /kaggle/input -name spatial_sgg_react.yaml | head -1))
echo "found data at: $INPUT"
mkdir -p $BASE/datasets $BASE/configs/hydra/Spatial
cp -r $INPUT/spatial_sgg $BASE/datasets/
cp -r $INPUT/spatial_sgg_yolo $BASE/datasets/
cp $INPUT/spatial_sgg_react.yaml $BASE/configs/hydra/Spatial/react.yaml
echo "DATA COPIED"; ls $BASE/datasets

found data at: /kaggle/input/datasets/shah9212/spatial-sgg
DATA COPIED
psg
spatial_sgg
spatial_sgg_yolo
vg


In [4]:
import json, glob, os
os.chdir("/kaggle/working/SGG-Benchmark")
for p in sorted(glob.glob("datasets/spatial_sgg/*/_annotations.human.coco.json") +
                glob.glob("datasets/spatial_sgg/*/_annotations.auto.coco.json")):
    d = json.load(open(p))
    if any(c["name"] == "__background__" for c in d["categories"]):
        continue
    for c in d["categories"]:      c["id"] += 1
    for c in d["rel_categories"]:  c["id"] += 1
    for a in d["annotations"]:     a["category_id"]  += 1
    for r in d["rel_annotations"]: r["predicate_id"] += 1
    d["categories"].insert(0, {"id": 0, "name": "__background__", "supercategory": "none"})
    d["rel_categories"].insert(0, {"id": 0, "name": "__no_relation__"})
    json.dump(d, open(p, "w"))

d = json.load(open("datasets/spatial_sgg/test/_annotations.human.coco.json"))
assert d["categories"][0]["name"] == "__background__" and len(d["categories"]) == 7
assert d["rel_categories"][0]["name"] == "__no_relation__" and len(d["rel_categories"]) == 8
print("PATCH OK - 7 object classes (bg+6), 8 relation classes (norel+7)")

PATCH OK - 7 object classes (bg+6), 8 relation classes (norel+7)


In [5]:
import glob, os, shutil
os.chdir("/kaggle/working/SGG-Benchmark")

hits = glob.glob("/kaggle/input/**/yolov8m_spatial.pt", recursive=True)
assert hits, ("detector checkpoint not found under /kaggle/input - attach the "
              "committed output of the original training notebook as an input")
os.makedirs("checkpoints/BACKBONES", exist_ok=True)
shutil.copy(hits[0], "checkpoints/BACKBONES/yolov8m_spatial.pt")
size = os.path.getsize("checkpoints/BACKBONES/yolov8m_spatial.pt") / 1e6
print(f"DETECTOR REUSED from {hits[0]}  ({size:.1f} MB)")
print("all arms and seeds share this frozen backbone, as in the first run")

DETECTOR REUSED from /kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/checkpoints/BACKBONES/yolov8m_spatial.pt  (52.0 MB)
all arms and seeds share this frozen backbone, as in the first run


In [6]:
import subprocess, shutil, os, re, time
os.chdir("/kaggle/working/SGG-Benchmark")

def stage(variant):
    """Point _annotations.coco.json at the chosen label source.
    Test is ALWAYS human gold in both arms: that is the design guarantee."""
    for split in ["train", "val"]:
        shutil.copy(f"datasets/spatial_sgg/{split}/_annotations.{variant}.coco.json",
                    f"datasets/spatial_sgg/{split}/_annotations.coco.json")
    shutil.copy("datasets/spatial_sgg/test/_annotations.human.coco.json",
                "datasets/spatial_sgg/test/_annotations.coco.json")

for seed in [43, 44]:
    for variant in ["human", "auto"]:
        tag = f"react_{variant}_s{seed}"
        stage(variant)
        os.system(f"rm -rf checkpoints/spatial/{tag}")
        cmd = ("python -u tools/relation_train_net_hydra.py "
               "--config-path ../configs/hydra/Spatial --config-name react "
               f"--task sgdet --save-best seed={seed} "
               f"output_dir=./checkpoints/spatial/{tag}")
        t0 = time.time()
        print("=" * 78, f"\nTRAIN {tag}\n", "=" * 78, flush=True)
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        out = r.stdout + "\n" + r.stderr
        print(out[-1500:], flush=True)
        mrs = [float(x) for x in re.findall(r"Result for mR:\s*([\d.]+)", out)]
        best = max(mrs) if mrs else 0.0
        print(f">>> {tag}: best val mR {best:.4f} in {(time.time()-t0)/60:.1f} min",
              flush=True)
        assert best > 0, f"{tag} produced mR=0 - check the class-index patch"

print("\nALL FOUR TRAINING RUNS DONE")

TRAIN react_human_s43
<00:02, 26.73it/s]
100%|██████████| 100/100 [00:03<00:00, 25.37it/s]

SGG Eval: 100%|██████████| 100/100 [00:00<00:00, 214.95it/s]

>>> react_human_s43: best val mR 0.1382 in 15.6 min
TRAIN react_auto_s43
0:02<00:01, 27.01it/s]
100%|██████████| 100/100 [00:03<00:00, 25.63it/s]

SGG Eval: 100%|██████████| 100/100 [00:01<00:00, 60.40it/s]

>>> react_auto_s43: best val mR 0.2150 in 26.8 min
TRAIN react_human_s44
<00:02, 26.79it/s]
100%|██████████| 100/100 [00:03<00:00, 25.86it/s]

SGG Eval: 100%|██████████| 100/100 [00:00<00:00, 210.15it/s]

>>> react_human_s44: best val mR 0.1384 in 11.7 min
TRAIN react_auto_s44
 67/100 [00:02<00:01, 26.48it/s]
100%|██████████| 100/100 [00:03<00:00, 25.79it/s]

SGG Eval: 100%|██████████| 100/100 [00:01<00:00, 66.07it/s]

>>> react_auto_s44: best val mR 0.2044 in 25.9 min

ALL FOUR TRAINING RUNS DONE


In [7]:
import subprocess, os, torch, logging, glob, shutil
os.chdir("/kaggle/working/SGG-Benchmark")
from omegaconf import OmegaConf
from sgg_benchmark.modeling.detector import build_detection_model
from sgg_benchmark.utils.checkpoint import DetectronCheckpointer
from sgg_benchmark.data import make_data_loader
from sgg_benchmark.engine.inference import inference

try:
    from sgg_benchmark.utils.logger import setup_logger
    logger = setup_logger("sgg_benchmark", ".", 0, verbose="INFO", steps=True)
except Exception:
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("sgg_benchmark")

# every arm is scored against human gold
shutil.copy("datasets/spatial_sgg/test/_annotations.human.coco.json",
            "datasets/spatial_sgg/test/_annotations.coco.json")

for seed in [43, 44]:
    for variant in ["human", "auto"]:
        tag = f"react_{variant}_s{seed}"
        cfg = OmegaConf.load(f"checkpoints/spatial/{tag}/hydra_config.yaml")
        out = f"./checkpoints/spatial/eval_{tag}"
        os.makedirs(out, exist_ok=True)
        cfg.output_dir = out
        ckpt = sorted(glob.glob(f"checkpoints/spatial/{tag}/best_model_epoch_*.pth"))[-1]

        model = build_detection_model(cfg).to(cfg.model.device)
        DetectronCheckpointer(cfg, model).load(ckpt)
        model.eval()
        loader = make_data_loader(cfg, mode="test")[0]

        print("=" * 80, flush=True)
        print(f"EVAL {tag} | {os.path.basename(ckpt)} | test images: {len(loader.dataset)}",
              flush=True)
        print("=" * 80, flush=True)
        with torch.no_grad():
            inference(cfg, model, loader, dataset_name="SpatialRobot_test",
                      iou_types=("bbox", "relations"), box_only=False,
                      device=cfg.model.device, expected_results=[],
                      expected_results_sigma_tol=4, output_folder=out, logger=logger)

print("\nALL EVALUATIONS DONE")

Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

100%|██████████| 210/210 [00:08<00:00, 24.11it/s]

2026-07-27 20:05:41,683 sgg_benchmark INFO: Total run time: 0:00:08 (38.95987706865583 ms / img per device, on 1 devices)
2026-07-27 20:05:41,684 sgg_benchmark INFO: Average latency per image: 38.95987706865583ms
2026-07-27 20:05:41,684 sgg_benchmark INFO: Standard deviation of latency: 51.093569346922656ms


2026-07-27 20:05:41,749 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-07-27 20:05:41,750 sgg_benchmark.data.build INFO: get dataset statistics...
2026-07-27 20:05:41,751 sgg_benchmark.data.build INFO: Loading data statistics from: ./checkpoints/spatial/eval_react_human_s43/SpatialRobot_statistics.cache
2026-07-27 20:05:41,752 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-07-27 20:05:41,755 sgg_benchmark INFO: Dynamically loaded 213 seen triplets from training statistics for zero-shot evaluation.
creating index...
index created!
Loading and preparing results...
Converting ndarray to lists...
(7390, 7)
0/7390
DONE (t=0.03s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.82s).
Accumulating evaluation results...
DONE (t=0.13s).
 Average Precision  (AP)

SGG Eval: 100%|██████████| 210/210 [00:01<00:00, 167.81it/s]

...

100%|██████████| 210/210 [00:07<00:00, 26.39it/s]

2026-07-27 20:06:29,978 sgg_benchmark INFO: Total run time: 0:00:07 (35.24876825241815 ms / img per device, on 1 devices)
2026-07-27 20:06:29,980 sgg_benchmark INFO: Average latency per image: 35.24876825241815ms
2026-07-27 20:06:29,980 sgg_benchmark INFO: Standard deviation of latency: 3.7636877274689993ms


2026-07-27 20:06:30,046 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-07-27 20:06:30,047 sgg_benchmark.data.build INFO: get dataset statistics...
2026-07-27 20:06:30,048 sgg_benchmark.data.build INFO: Loading data statistics from: ./checkpoints/spatial/eval_react_auto_s44/SpatialRobot_statistics.cache
2026-07-27 20:06:30,048 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-07-27 20:06:30,052 sgg_benchmark INFO: Dynamically loaded 213 seen triplets from training statistics for zero-shot evaluation.
creating index...
index created!
Loading and preparing results...
Converting ndarray to lists...
(7390, 7)
0/7390
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.70s).
Accumulating evaluation results...
DONE (t=0.13s).
 Average Precision  (AP) 

SGG Eval: 100%|██████████| 210/210 [00:01<00:00, 190.29it/s]

2026-07-27 20:06:32,152 sgg_benchmark INFO: 
Detection evaluation mAp=0.6541
SGG eval:     R @ 20: 0.1396;     R @ 50: 0.2030;     R @ 100: 0.2505;  for mode=sgdet.
SGG eval:    mR @ 20: 0.1672;    mR @ 50: 0.2233;    mR @ 100: 0.2681;  for mode=sgdet.
----------------------- Details ------------------------
(on:0.6392) (under:0.6932) (to the left of:0.1545) (to the right of:0.1770) (in front of:0.0710) (behind:0.1420) (near:0.0000) 
--------------------------------------------------------
SGG eval:     zR @ 20: 0.0000;     zR @ 50: 0.0000;     zR @ 100: 0.0000;  for mode=sgdet.
SGG eval:     F1 @ 20: 0.1522;     F1 @ 50: 0.2127;     F1 @ 100: 0.2590;  for mode=sgdet.


ALL EVALUATIONS DONE


In [8]:
%%bash
cd /kaggle/working/SGG-Benchmark
zip -rq /kaggle/working/seed_results.zip checkpoints/spatial -x "*.pth" -x "*.pt"
echo "RESULTS -> /kaggle/working/seed_results.zip"
ls -la /kaggle/working/seed_results.zip

RESULTS -> /kaggle/working/seed_results.zip
-rw-r--r-- 1 root root 145282 Jul 27 20:06 /kaggle/working/seed_results.zip
